In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
load_dotenv()

# ============================================================
# 1. STATE
# ============================================================

class Task(BaseModel):
    id: str
    title: str
    description: str
    dependencies: list[str] = Field(default_factory=list)
    status: str = "todo"


class StudyPlan(BaseModel):
    tasks: list[Task]


class StudyState(TypedDict):
    goal: str
    plan: list[dict]
    current_task: str | None
    completed_tasks: list[str]
    failed_tasks: list[str]
    message: str


# ============================================================
# 2. LLM
# ============================================================

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL"),
    temperature=0.3,
)



# ============================================================
# 3. PLANNER NODE
# ============================================================

def planner(state: StudyState):
    """
    Takes the user's goal and creates a task plan.
    """

    prompt = f"""
You are a study planning agent.

Create a practical learning plan for:

{state["goal"]}

Rules:
- Create 5-8 tasks.
- Tasks should be small and achievable.
- Identify dependencies between tasks.
- A task should depend on another task when the previous knowledge
  is needed first.
- Give every task a unique ID such as T1, T2, T3.
"""

    structured_llm = llm.with_structured_output(StudyPlan)

    result = structured_llm.invoke(prompt)

    plan = [task.model_dump() for task in result.tasks]

    return {
        "plan": plan,
        "message": "Study plan created."
    }


# ============================================================
# 4. SELECT NEXT TASK
# ============================================================

def select_task(state: StudyState):
    """
    Find the next task whose dependencies are complete.
    """

    completed = set(state["completed_tasks"])

    for task in state["plan"]:

        # Skip completed/failed tasks
        if task["status"] != "todo":
            continue

        dependencies = task["dependencies"]

        # Check dependencies
        if all(dep in completed for dep in dependencies):

            return {
                "current_task": task["id"],
                "message": f"Selected task: {task['title']}"
            }

    return {
        "current_task": None,
        "message": "No more tasks available."
    }


# ============================================================
# 5. EXECUTE TASK
# ============================================================

def execute_task(state: StudyState):
    """
    Simulate execution of the study task.

    Later we can replace this with actual tools:
    - web search
    - quiz generator
    - documentation search
    - coding exercises
    """

    task_id = state["current_task"]

    if not task_id:
        return {
            "message": "Nothing to execute."
        }

    print(f"\n📚 Executing {task_id}")

    # Find task
    task = next(
        task for task in state["plan"]
        if task["id"] == task_id
    )

    print(f"Task: {task['title']}")
    print(f"Description: {task['description']}")

    # --------------------------------------------------------
    # Simulated execution
    #
    # For now we assume success.
    # Later we will make this intelligent.
    # --------------------------------------------------------

    success = True

    if success:

        # Update task status
        for t in state["plan"]:
            if t["id"] == task_id:
                t["status"] = "done"

        return {
            "completed_tasks": state["completed_tasks"] + [task_id],
            "message": f"{task_id} completed successfully."
        }

    else:

        for t in state["plan"]:
            if t["id"] == task_id:
                t["status"] = "failed"

        return {
            "failed_tasks": state["failed_tasks"] + [task_id],
            "message": f"{task_id} failed."
        }


# ============================================================
# 6. DECIDE WHAT TO DO NEXT
# ============================================================

def decide_next(state: StudyState):
    """
    Decide whether to continue, re-plan, or finish.
    """

    # No task left to execute
    if state["current_task"] is None:
        return "finish"

    # All tasks completed
    if len(state["completed_tasks"]) == len(state["plan"]):
        return "finish"

    # Current task failed
    if state["current_task"] in state["failed_tasks"]:
        return "replan"

    # Otherwise continue
    return "continue"


# ============================================================
# 7. RE-PLANNER
# ============================================================

def replan(state: StudyState):
    """
    Very simple re-planning node.

    We'll make this much smarter later.
    """

    failed_task = state["current_task"]

    print(f"\n🔄 Re-planning because {failed_task} failed.")

    # For now, just reset the failed task.
    for task in state["plan"]:
        if task["id"] == failed_task:
            task["status"] = "todo"

    return {
        "message": f"Re-planned task {failed_task}."
    }


# ============================================================
# 8. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(StudyState)


# Nodes
builder.add_node("planner", planner)
builder.add_node("select_task", select_task)
builder.add_node("execute_task", execute_task)
builder.add_node("replan", replan)


# Starting point
builder.add_edge(START, "planner")


# Planner → task selection
builder.add_edge("planner", "select_task")

# Task selection → execution
builder.add_edge("select_task", "execute_task")

# Execution → decision
builder.add_conditional_edges(
    "execute_task",
    decide_next,
    {
        "continue": "select_task",
        "replan": "replan",
        "finish": END,
    },
)


# Re-plan → select another task
builder.add_edge("replan", "select_task")

# Compile graph
graph = builder.compile()




In [6]:

# ============================================================
# 9. RUN THE AGENT
# ============================================================


goal = "I want to learn Agentic AI"

initial_state = {
    "goal": goal,
    "plan": [],
    "current_task": None,
    "completed_tasks": [],
    "failed_tasks": [],
    "message": "",
}

print("\n🚀 Starting Study Agent...\n")

final_state = graph.invoke(initial_state, config={"recursion_limit": 50})

print("\n" + "=" * 50)
print("FINAL STUDY PLAN")
print("=" * 50)



🚀 Starting Study Agent...


FINAL STUDY PLAN


In [7]:

for task in final_state["plan"]:
    print(
        f"\n{task['id']}: {task['title']}"
        f"\n   Status: {task['status']}"
        f"\n   Depends on: {task['dependencies']}"
    )

print("\nCompleted:", final_state["completed_tasks"])


T1: Understand Basic Concepts of AI and Machine Learning
   Status: Not Started
   Depends on: []

T2: Learn About the Role of Agents in AI Systems
   Status: Not Started
   Depends on: ['T1']

T3: Explore Key Concepts of Agentic AI
   Status: Not Started
   Depends on: ['T2']

T4: Analyze Case Studies of Agentic AI Applications
   Status: Not Started
   Depends on: ['T3']

T5: Develop a Simple Agentic AI Model
   Status: Not Started
   Depends on: ['T4']

T6: Implement Ethical Considerations in Agentic AI Design
   Status: Not Started
   Depends on: ['T5']

T7: Present Your Agentic AI Model and Ethical Considerations
   Status: Not Started
   Depends on: ['T6']

Completed: []
